# Warehouse Slot Ranking Model

Notebook Colab này được chỉnh để khớp với dự án `warehouse-service`:
- bám theo domain `optimization` hiện có
- dùng hard rules giống logic trong backend
- sinh synthetic data theo schema kho / slot / container của dự án
- train LightGBM Ranker để xếp hạng slot ứng viên
- export CSV và lưu model

## Mục tiêu
Đề xuất vị trí container tốt nhất dựa trên:
- ngày xuất
- công ty / cụm công ty
- cân nặng
- loại hàng / loại kho
- kích thước 20ft / 40ft
- ràng buộc yard và khả năng truy xuất


## 1. Cài đặt thư viện
Nếu đang chạy trên Colab, cell này sẽ cài dependency cần thiết.


In [ ]:
!pip -q install lightgbm pandas numpy scikit-learn joblib matplotlib seaborn

## 2. Import thư viện và cấu hình
Khởi tạo random seed để kết quả có thể tái lập.


In [ ]:
import math
import random
from datetime import datetime, timedelta
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import files
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
random.seed(42)
np.random.seed(42)

OUTPUT_DIR = Path('/content/warehouse_ranker आउट')
OUTPUT_DIR = Path('/content/warehouse_ranker_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## 3. Schema dataset
Notebook dùng 3 nhóm dữ liệu:
- `containers`
- `slots`
- `yard_state`

Mỗi sample train là một cặp `container + candidate_slot`.


In [ ]:
CONTAINER_COLUMNS = [
    'container_id', 'company', 'export_date', 'import_date', 'weight_kg',
    'size_type', 'cargo_type', 'status', 'priority_level', 'is_damaged',
    'special_handling', 'warehouse_type_required', 'estimated_dwell_days'
]

SLOT_COLUMNS = [
    'slot_id', 'warehouse_id', 'zone_id', 'floor_no', 'position_no',
    'slot_type', 'allowed_cargo_type', 'allowed_size_type', 'max_weight_kg',
    'current_occupancy', 'stack_height', 'is_locked', 'distance_to_exit',
    'is_reserved', 'is_temporary_buffer', 'is_cold_warehouse',
    'is_damaged_warehouse', 'is_40ft_designated'
]

# Match backend optimizer concepts more closely
FEATURE_COLUMNS = [
    'warehouse_id', 'zone_id', 'floor_no', 'position_no', 'slot_type_40ft',
    'container_size_40ft', 'container_weight_kg', 'container_priority',
    'container_damaged', 'export_urgency_days', 'company_match', 'weight_fit',
    'accessibility', 'move_reduction', 'zone_compatibility', 'blocking_risk',
    'occupancy_rate', 'stack_ratio', 'is_cold_warehouse', 'is_damaged_warehouse',
    'is_40ft_designated'
]

print('Container columns:', CONTAINER_COLUMNS)
print('Slot columns:', SLOT_COLUMNS)
print('Feature columns:', FEATURE_COLUMNS)

## 4. Tạo synthetic data
Mô phỏng layout cố định:
- 4 kho
- 3 zone / kho
- 4 tầng / zone
- 16 slot 20ft + 8 slot 40ft / tầng


In [ ]:
WAREHOUSE_TYPES = ['dry', 'cold', 'fragile', 'hazard']
CARGO_TYPES = ['dry', 'cold', 'fragile', 'hazard']
SIZE_TYPES = ['20ft', '40ft']
COMPANIES = ['A Logistics', 'B Shipping', 'C Trade', 'D Cargo', 'E Global']
STATUSES = ['NEW', 'IN_YARD', 'READY_TO_EXIT', 'HOLD']

def generate_slots():
    rows = []
    slot_id = 1
    for warehouse_id in range(1, 5):
        warehouse_type = WAREHOUSE_TYPES[(warehouse_id - 1) % len(WAREHOUSE_TYPES)]
        for zone_id in range(1, 4):
            for floor_no in range(1, 5):
                # 16 slot cho 20ft / tầng
                for pos in range(1, 17):
                    rows.append({
                        'slot_id': f'S{slot_id:05d}',
                        'warehouse_id': warehouse_id,
                        'zone_id': zone_id,
                        'floor_no': floor_no,
                        'position_no': pos,
                        'slot_type': '20ft',
                        'allowed_cargo_type': warehouse_type,
                        'allowed_size_type': '20ft',
                        'max_weight_kg': int(28000 + (4 - floor_no) * 5000 + np.random.randint(-1000, 1000)),
                        'current_occupancy': int(np.random.randint(0, 2)),
                        'stack_height': int(np.random.randint(0, floor_no + 1)),
                        'is_locked': bool(np.random.rand() < 0.03),
                        'distance_to_exit': int(np.random.randint(1, 100)),
                        'is_reserved': bool(np.random.rand() < 0.05),
                        'is_temporary_buffer': False,
                        'is_cold_warehouse': warehouse_type == 'cold',
                        'is_damaged_warehouse': warehouse_type == 'fragile',
                        'is_40ft_designated': False,
                    })
                    slot_id += 1
                # 8 slot dành cho 40ft / tầng
                for pos in range(1, 9):
                    rows.append({
                        'slot_id': f'S{slot_id:05d}',
                        'warehouse_id': warehouse_id,
                        'zone_id': zone_id,
                        'floor_no': floor_no,
                        'position_no': pos,
                        'slot_type': '40ft',
                        'allowed_cargo_type': warehouse_type,
                        'allowed_size_type': '40ft',
                        'max_weight_kg': int(32000 + (4 - floor_no) * 6000 + np.random.randint(-1000, 1000)),
                        'current_occupancy': int(np.random.randint(0, 2)),
                        'stack_height': int(np.random.randint(0, floor_no + 1)),
                        'is_locked': bool(np.random.rand() < 0.03),
                        'distance_to_exit': int(np.random.randint(1, 100)),
                        'is_reserved': bool(np.random.rand() < 0.05),
                        'is_temporary_buffer': False,
                        'is_cold_warehouse': warehouse_type == 'cold',
                        'is_damaged_warehouse': warehouse_type == 'fragile',
                        'is_40ft_designated': True,
                    })
                    slot_id += 1
    return pd.DataFrame(rows)

def generate_containers(n=800):
    today = datetime.today().date()
    rows = []
    for i in range(n):
        cargo_type = np.random.choice(CARGO_TYPES, p=[0.55, 0.15, 0.15, 0.15])
        size_type = np.random.choice(SIZE_TYPES, p=[0.7, 0.3])
        export_delta = int(np.random.randint(0, 21))
        import_delta = int(np.random.randint(1, 20))
        export_date = today + timedelta(days=export_delta)
        import_date = today - timedelta(days=import_delta)
        weight = int(np.clip(np.random.normal(18000 if size_type == '20ft' else 25000, 4500), 2000, 35000))
        rows.append({
            'container_id': f'C{i:06d}',
            'company': np.random.choice(COMPANIES),
            'export_date': export_date,
            'import_date': import_date,
            'weight_kg': weight,
            'size_type': size_type,
            'cargo_type': cargo_type,
            'status': np.random.choice(STATUSES),
            'priority_level': np.random.choice([1, 2, 3], p=[0.5, 0.35, 0.15]),
            'is_damaged': bool(np.random.rand() < (0.05 if cargo_type != 'hazard' else 0.02)),
            'special_handling': bool(np.random.rand() < 0.1),
            'warehouse_type_required': cargo_type,
            'estimated_dwell_days': int(np.clip(export_delta + np.random.normal(0, 2), 0, 30)),
        })
    return pd.DataFrame(rows)

slots_df = generate_slots()
containers_df = generate_containers()

print('slots:', slots_df.shape)
print('containers:', containers_df.shape)
display(slots_df.head())
display(containers_df.head())

## 5. Hard rules
Slot phải hợp lệ trước khi đưa vào ranking.


In [ ]:
def is_valid_slot(container, slot):
    if slot['is_locked'] or slot['is_reserved']:
        return False

    # hard rule: đúng kho đúng loại hàng
    if container['cargo_type'] != slot['allowed_cargo_type']:
        return False

    # hard rule: size
    if container['size_type'] != slot['allowed_size_type']:
        if not (container['size_type'] == '20ft' and slot['slot_type'] == '40ft' and slot['is_40ft_designated']):
            return False

    # hard rule: tải trọng
    if container['weight_kg'] > slot['max_weight_kg']:
        return False

    # ngoại lệ backend: cold warehouse không dùng cho hàng hỏng
    if container['is_damaged'] and slot['is_cold_warehouse']:
        return False

    return True

def get_candidate_slots(container_row, slots):
    mask = slots.apply(lambda s: is_valid_slot(container_row, s), axis=1)
    return slots[mask].copy()

example_candidates = get_candidate_slots(containers_df.iloc[0], slots_df)
print('candidate slots for sample container:', len(example_candidates))
display(example_candidates.head())

## 6. Feature engineering
Tạo feature cho từng cặp container-slot.


In [ ]:
def build_features(container, slot, yard_state=None):
    export_urgency_days = max(0, (container['export_date'] - datetime.today().date()).days)
    export_urgency = 1.0 / (1.0 + export_urgency_days)

    # Company clustering proxy: same company hash -> same zone preference
    company_match = 1 if slot['zone_id'] == ((abs(hash(container['company'])) % 3) + 1) else 0

    weight_fit = 1.0 - min(abs(container['weight_kg'] - slot['max_weight_kg']) / max(slot['max_weight_kg'], 1), 1.0)
    accessibility = 1.0 - min(slot['distance_to_exit'] / 100.0, 1.0)
    move_reduction = 1.0 - min(slot['stack_height'] / 4.0, 1.0)
    zone_compatibility = 1.0 if container['cargo_type'] == slot['allowed_cargo_type'] else 0.0
    blocking_risk = min((slot['stack_height'] / 4.0) + (slot['current_occupancy'] / 2.0), 1.0)
    occupancy_rate = min(slot['current_occupancy'] / 2.0, 1.0)
    stack_ratio = min(slot['stack_height'] / 4.0, 1.0)

    if yard_state is None:
        yard_state = {}

    return {
        'warehouse_id': slot['warehouse_id'],
        'zone_id': slot['zone_id'],
        'floor_no': slot['floor_no'],
        'position_no': slot['position_no'],
        'slot_type_40ft': 1 if slot['slot_type'] == '40ft' else 0,
        'container_size_40ft': 1 if container['size_type'] == '40ft' else 0,
        'container_weight_kg': container['weight_kg'],
        'container_priority': container['priority_level'],
        'container_damaged': int(container['is_damaged']),
        'export_urgency_days': export_urgency_days,
        'company_match': company_match,
        'weight_fit': weight_fit,
        'accessibility': accessibility,
        'move_reduction': move_reduction,
        'zone_compatibility': zone_compatibility,
        'blocking_risk': blocking_risk,
        'occupancy_rate': occupancy_rate,
        'stack_ratio': stack_ratio,
        'is_cold_warehouse': int(slot['is_cold_warehouse']),
        'is_damaged_warehouse': int(slot['is_damaged_warehouse']),
        'is_40ft_designated': int(slot['is_40ft_designated']),
    }

sample_feat = build_features(containers_df.iloc[0], example_candidates.iloc[0])
sample_feat

## 7. Tạo training dataset
Ta sinh nhiều candidate cho mỗi container, sau đó gán label theo heuristic score.


In [ ]:
WEIGHTS = {
    'export_urgency': 0.28,
    'company_match': 0.12,
    'weight_fit': 0.18,
    'accessibility': 0.14,
    'move_reduction': 0.12,
    'zone_compatibility': 0.10,
    'blocking_risk': -0.16,
}

def heuristic_score(row):
    return sum(row[k] * w for k, w in WEIGHTS.items())

def build_dataset(containers, slots, max_candidates=120):
    rows = []
    for _, c in containers.iterrows():
        candidates = get_candidate_slots(c, slots)
        if len(candidates) == 0:
            continue
        candidates = candidates.sample(n=min(max_candidates, len(candidates)), random_state=42)
        yard_state = {
            'occupancy_rate': float(candidates['current_occupancy'].mean() / 2.0),
            'blocked_risk': float(np.clip(candidates['stack_height'].mean() / 4.0, 0, 1)),
            'company_cluster_score': 0.5,
        }
        scored = []
        for _, s in candidates.iterrows():
            feat = build_features(c, s, yard_state)
            score = heuristic_score(feat)
            scored.append((feat, score, s['slot_id']))

        # create multi-level label: 3=best, 2=good, 1=acceptable, 0=others
        scores = [x[1] for x in scored]
        best = max(scores)
        p75 = np.percentile(scores, 75)
        p50 = np.percentile(scores, 50)

        for feat, score, slot_id in scored:
            feat['container_id'] = c['container_id']
            feat['slot_id'] = slot_id
            if score >= best - 1e-9:
                label = 3
            elif score >= p75:
                label = 2
            elif score >= p50:
                label = 1
            else:
                label = 0
            feat['label'] = int(label)
            feat['relevance'] = score
            rows.append(feat)

    return pd.DataFrame(rows)

train_df = build_dataset(containers_df, slots_df)
print('train_df shape:', train_df.shape)
display(train_df.head())

## 8. Export CSV dataset
Cell này xuất ra CSV để bạn tải về hoặc dùng cho lần train khác.


In [ ]:
containers_path = OUTPUT_DIR / 'containers_synthetic.csv'
slots_path = OUTPUT_DIR / 'slots_synthetic.csv'
dataset_path = OUTPUT_DIR / 'ranker_training_dataset.csv'

containers_df.to_csv(containers_path, index=False)
slots_df.to_csv(slots_path, index=False)
train_df.to_csv(dataset_path, index=False)

print('Saved:')
print(containers_path)
print(slots_path)
print(dataset_path)

# Nút tải file về máy
files.download(str(containers_path))
files.download(str(slots_path))
files.download(str(dataset_path))

## 9. Train / test split
Split theo `container_id` để tránh leakage.


In [ ]:
container_ids = train_df['container_id'].unique()
train_ids, test_ids = train_test_split(container_ids, test_size=0.2, random_state=42)
train_set = train_df[train_df['container_id'].isin(train_ids)].copy()
test_set = train_df[train_df['container_id'].isin(test_ids)].copy()

print('train_set:', train_set.shape)
print('test_set:', test_set.shape)
print('train containers:', train_set['container_id'].nunique())
print('test containers:', test_set['container_id'].nunique())

## 10. Train LightGBM Ranker
Model objective là `lambdarank`.


In [ ]:
X_train = train_set[FEATURE_COLUMNS]
y_train = train_set['label'].astype(int)
group_train = train_set.groupby('container_id').size().tolist()

X_test = test_set[FEATURE_COLUMNS]
y_test = test_set['label'].astype(int)
group_test = test_set.groupby('container_id').size().tolist()

ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    boosting_type='gbdt',
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=42,
    label_gain=[0, 1, 3, 7],
)

ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_test, y_test)],
    eval_group=[group_test],
    eval_at=[1, 3, 5],
    eval_metric='ndcg',
)

print('Training completed')

## 11. Đánh giá model
Tính NDCG@5 trên test set.


In [ ]:
def evaluate_ndcg_at_k(df, model, k=5):
    scores = []
    for cid, g in df.groupby('container_id'):
        X = g[FEATURE_COLUMNS]
        y_true = g['label'].astype(int).values.reshape(1, -1)
        y_score = model.predict(X).reshape(1, -1)
        scores.append(ndcg_score(y_true, y_score, k=min(k, len(g))))
    return float(np.mean(scores)) if scores else 0.0

ndcg1 = evaluate_ndcg_at_k(test_set, ranker, k=1)
ndcg3 = evaluate_ndcg_at_k(test_set, ranker, k=3)
ndcg5 = evaluate_ndcg_at_k(test_set, ranker, k=5)
print(f'NDCG@1 = {ndcg1:.4f}')
print(f'NDCG@3 = {ndcg3:.4f}')
print(f'NDCG@5 = {ndcg5:.4f}')

## 12. Biểu đồ nhanh
Xem phân phối score và feature importance.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(train_df['label'], bins=4, discrete=True)
plt.title('Distribution of ranking labels')
plt.show()

fi = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': ranker.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi.head(12), x='importance', y='feature')
plt.title('Top feature importances')
plt.show()

display(fi)

## 13. Xem top-5 slot cho 1 container mẫu
Lấy container đầu tiên trong test set và sort theo score dự đoán.


In [ ]:
sample_cid = test_set['container_id'].iloc[0]
sample_rows = test_set[test_set['container_id'] == sample_cid].copy()
sample_rows['pred_score'] = ranker.predict(sample_rows[FEATURE_COLUMNS])
sample_rows = sample_rows.sort_values('pred_score', ascending=False)

display(sample_rows[['container_id', 'slot_id', 'pred_score', 'label', 'warehouse_id', 'zone_id', 'floor_no', 'position_no']].head(5))

## 14. Lưu model
Lưu cả model LightGBM và metadata của feature list.


In [ ]:
model_path = OUTPUT_DIR / 'warehouse_ranker_lgbm.txt'
meta_path = OUTPUT_DIR / 'warehouse_ranker_meta.joblib'

ranker.booster_.save_model(str(model_path))
joblib.dump({
    'feature_columns': FEATURE_COLUMNS,
    'weights': WEIGHTS,
    'ndcg1': ndcg1,
    'ndcg3': ndcg3,
    'ndcg5': ndcg5,
    'label_gain': [0, 1, 3, 7],
}, meta_path)

print('Model saved to:')
print(model_path)
print(meta_path)

files.download(str(model_path))
files.download(str(meta_path))

## 15. Ghi chú mở rộng

Các bước nâng cấp tiếp theo:
- thay heuristic label bằng dữ liệu lịch sử thật
- thêm blocking risk, relocation history, company clustering thực
- tách hard-rule filter thành module riêng
- xuất model thành API service để backend Spring Boot gọi
